# Token Usage by User - View & Dashboard Setup

This notebook:
1. Creates a view `token_usage_by_user` under your chosen catalog/schema
2. Creates an AI/BI dashboard named **Token Usage Per User** from that view

## Step 1: Configure Catalog and Schema

In [0]:
dbutils.widgets.text("catalog", "main", "1. Catalog")
dbutils.widgets.text("schema", "default", "2. Schema")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
print(f"Target: `{catalog}`.`{schema}`.token_usage_by_user")

Target: `shao_sandbox1`.`default`.token_usage_by_user


## Step 2: Create the View

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

spark.sql(f"""
CREATE OR REPLACE VIEW `{catalog}`.`{schema}`.token_usage_by_user AS
WITH combined AS (
  SELECT
    DATE(u.event_time) as usage_date,
    u.requester,
    u.endpoint_name,
    u.destination_model as model,
    COALESCE(w.workspace_name, u.workspace_id) as workspace_name,
    SUM(COALESCE(u.input_tokens, 0)) as input_tokens,
    SUM(COALESCE(u.output_tokens, 0)) as output_tokens,
    SUM(COALESCE(u.input_tokens, 0) + COALESCE(u.output_tokens, 0)) as total_tokens,
    COUNT(*) as request_count
  FROM
    `system`.`ai_gateway`.`usage` u
      LEFT JOIN `system`.`access`.`workspaces_latest` w
        ON u.workspace_id = w.workspace_id
  GROUP BY
    usage_date,
    u.requester,
    u.endpoint_name,
    model,
    COALESCE(w.workspace_name, u.workspace_id)
  UNION ALL
  SELECT
    DATE(eu.request_time) as usage_date,
    eu.requester,
    se.endpoint_name,
    LOWER(regexp_replace(se.entity_name, '[ .]+', '-')) as model,
    COALESCE(w2.workspace_name, eu.workspace_id) as workspace_name,
    SUM(COALESCE(eu.input_token_count, 0)) as input_tokens,
    SUM(COALESCE(eu.output_token_count, 0)) as output_tokens,
    SUM(COALESCE(eu.input_token_count, 0) + COALESCE(eu.output_token_count, 0)) as total_tokens,
    COUNT(*) as request_count
  FROM
    `system`.`serving`.`endpoint_usage` eu
      JOIN `system`.`serving`.`served_entities` se
        ON eu.served_entity_id = se.served_entity_id
      LEFT JOIN `system`.`access`.`workspaces_latest` w2
        ON eu.workspace_id = w2.workspace_id
  GROUP BY
    usage_date,
    eu.requester,
    se.endpoint_name,
    model,
    COALESCE(w2.workspace_name, eu.workspace_id)
),
enriched AS (
  SELECT
    *,
    SUM(total_tokens) OVER (PARTITION BY endpoint_name, usage_date) as ep_total_tokens
  FROM combined
),
user_endpoint_agg AS (
  SELECT
    usage_date,
    requester,
    endpoint_name,
    model,
    workspace_name,
    SUM(input_tokens) as input_tokens,
    SUM(output_tokens) as output_tokens,
    SUM(total_tokens) as total_tokens,
    SUM(request_count) as request_count,
    ANY_VALUE(ep_total_tokens) as ep_total_tokens
  FROM enriched
  GROUP BY usage_date, requester, endpoint_name, model, workspace_name
),
billing AS (
  SELECT
    u.usage_metadata.endpoint_name AS endpoint_name,
    u.usage_date AS usage_date,
    ROUND(SUM(u.usage_quantity * lp.pricing.effective_list.default), 2) AS total_cost_usd
  FROM
    system.billing.usage u
      INNER JOIN system.billing.list_prices lp
        ON u.cloud = lp.cloud
        AND u.sku_name = lp.sku_name
        AND u.usage_start_time >= lp.price_start_time
        AND (
          u.usage_end_time <= lp.price_end_time
          OR lp.price_end_time IS NULL
        )
  WHERE
    u.record_type = 'ORIGINAL'
    AND u.sku_name IN (
      'ENTERPRISE_ANTHROPIC_MODEL_SERVING',
      'ENTERPRISE_OPENAI_MODEL_SERVING',
      'ENTERPRISE_GEMINI_MODEL_SERVING',
      'PREMIUM_ANTHROPIC_MODEL_SERVING',
      'PREMIUM_OPENAI_MODEL_SERVING',
      'PREMIUM_GEMINI_MODEL_SERVING'
    )
  GROUP BY
    u.usage_metadata.endpoint_name,
    u.usage_date
),
with_cost AS (
  SELECT
    uea.usage_date,
    uea.requester,
    uea.model,
    uea.workspace_name,
    uea.input_tokens,
    uea.output_tokens,
    uea.total_tokens,
    uea.request_count,
    uea.total_tokens * 1.0 / NULLIF(uea.ep_total_tokens, 0) * b.total_cost_usd as proxied_cost
  FROM
    user_endpoint_agg uea
      LEFT JOIN billing b
        ON uea.endpoint_name = b.endpoint_name
        AND uea.usage_date = b.usage_date
)
SELECT
  usage_date,
  requester,
  model,
  workspace_name,
  SUM(input_tokens) as input_tokens,
  SUM(output_tokens) as output_tokens,
  SUM(total_tokens) as total_tokens,
  SUM(request_count) as request_count,
  ROUND(SUM(proxied_cost), 2) as proxied_cost_usd
FROM
  with_cost
WHERE
  is_member('admins') OR requester = CURRENT_USER()
GROUP BY
  usage_date,
  requester,
  model,
  workspace_name
ORDER BY
  usage_date DESC,
  total_tokens DESC
""")

# Grant USE CATALOG and USE SCHEMA so account users can traverse the hierarchy
spark.sql(f"GRANT USE CATALOG ON CATALOG `{catalog}` TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA `{catalog}`.`{schema}` TO `account users`")
# Grant SELECT on the view
spark.sql(f"GRANT SELECT ON VIEW `{catalog}`.`{schema}`.token_usage_by_user TO `account users`")

print(f"View `{catalog}`.`{schema}`.token_usage_by_user created successfully!")
print("Note: Non-admin users will only see their own usage data.")

View `shao_sandbox1`.`default`.token_usage_by_user created successfully!
Note: Non-admin users will only see their own usage data.


In [0]:
# Verify the view
display(spark.sql(f"SELECT * FROM `{catalog}`.`{schema}`.token_usage_by_user LIMIT 10"))

## Step 3: Create the AI/BI Dashboard

In [0]:
import json
import requests

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
view_name = f"`{catalog}`.`{schema}`.`token_usage_by_user`"

host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

warehouse_id = "862f1d757f0424f7"
dashboard_name = "Token Usage Per User"

dashboard_json = {
    "datasets": [
        {
            "name": "token_usage",
            "displayName": "Token Usage By User",
            "queryLines": [
                "SELECT\n",
                "  requester,\n",
                "  model,\n",
                "  workspace_name,\n",
                "  SUM(input_tokens) AS input_tokens,\n",
                "  SUM(output_tokens) AS output_tokens,\n",
                "  SUM(total_tokens) AS total_tokens,\n",
                "  SUM(request_count) AS request_count,\n",
                "  SUM(proxied_cost_usd) AS proxied_cost_usd\n",
                "FROM\n",
                f"  {view_name}\n",
                "WHERE\n",
                "  usage_date >= :time_range.min\n",
                "  AND usage_date <= :time_range.max\n",
                "GROUP BY\n",
                "  requester,\n",
                "  model,\n",
                "  workspace_name\n",
                "ORDER BY\n",
                "  total_tokens DESC"
            ],
            "parameters": [
                {
                    "displayName": "Time Range",
                    "keyword": "time_range",
                    "dataType": "DATE",
                    "complexType": "RANGE",
                    "defaultSelection": {
                        "range": {
                            "dataType": "DATE",
                            "min": {"value": "now-30d/d"},
                            "max": {"value": "now/d"}
                        }
                    }
                }
            ]
        }
    ],
    "pages": [
        {
            "name": "top_users",
            "displayName": "Top Users",
            "layout": [
                {
                    "widget": {
                        "name": "filter-date-range",
                        "queries": [
                            {
                                "name": "parameter_dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_time_range",
                                "query": {
                                    "datasetName": "token_usage",
                                    "parameters": [{"name": "time_range", "keyword": "time_range"}],
                                    "disaggregated": False
                                }
                            }
                        ],
                        "spec": {
                            "version": 2,
                            "frame": {"title": "Time Range", "showTitle": True},
                            "widgetType": "filter-date-range-picker",
                            "encodings": {
                                "fields": [{"parameterName": "time_range", "queryName": "parameter_dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_time_range"}]
                            }
                        }
                    },
                    "position": {"x": 0, "y": 0, "width": 6, "height": 1}
                },
                {
                    "widget": {
                        "name": "filter-workspace",
                        "queries": [
                            {
                                "name": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_workspace_name",
                                "query": {
                                    "datasetName": "token_usage",
                                    "fields": [
                                        {"name": "workspace_name", "expression": "`workspace_name`"},
                                        {"name": "workspace_name_associativity", "expression": "COUNT_IF(`associative_filter_predicate_group`)"}
                                    ],
                                    "disaggregated": False
                                }
                            }
                        ],
                        "spec": {
                            "version": 2,
                            "frame": {"title": "Workspace", "showTitle": True},
                            "widgetType": "filter-multi-select",
                            "encodings": {
                                "fields": [{"fieldName": "workspace_name", "queryName": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_workspace_name"}]
                            }
                        }
                    },
                    "position": {"x": 6, "y": 0, "width": 6, "height": 1}
                },
                {
                    "widget": {
                        "name": "filter-user",
                        "queries": [
                            {
                                "name": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_requester",
                                "query": {
                                    "datasetName": "token_usage",
                                    "fields": [
                                        {"name": "requester", "expression": "`requester`"},
                                        {"name": "requester_associativity", "expression": "COUNT_IF(`associative_filter_predicate_group`)"}
                                    ],
                                    "disaggregated": False
                                }
                            }
                        ],
                        "spec": {
                            "version": 2,
                            "frame": {"title": "User", "showTitle": True},
                            "widgetType": "filter-multi-select",
                            "encodings": {
                                "fields": [{"fieldName": "requester", "queryName": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_requester"}]
                            }
                        }
                    },
                    "position": {"x": 0, "y": 1, "width": 6, "height": 1}
                },
                {
                    "widget": {
                        "name": "filter-model",
                        "queries": [
                            {
                                "name": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_model",
                                "query": {
                                    "datasetName": "token_usage",
                                    "fields": [
                                        {"name": "model", "expression": "`model`"},
                                        {"name": "model_associativity", "expression": "COUNT_IF(`associative_filter_predicate_group`)"}
                                    ],
                                    "disaggregated": False
                                }
                            }
                        ],
                        "spec": {
                            "version": 2,
                            "frame": {"title": "Model", "showTitle": True},
                            "widgetType": "filter-multi-select",
                            "encodings": {
                                "fields": [{"fieldName": "model", "queryName": "dashboards/01f1333dec521684b38bccf005ced954/datasets/01f1333dec521702849b45e6cbbe9341_model"}]
                            }
                        }
                    },
                    "position": {"x": 6, "y": 1, "width": 6, "height": 1}
                },
                {
                    "widget": {
                        "name": "top-users-table",
                        "queries": [
                            {
                                "name": "main_query",
                                "query": {
                                    "datasetName": "token_usage",
                                    "fields": [
                                        {"name": "requester", "expression": "`requester`"},
                                        {"name": "model", "expression": "`model`"},
                                        {"name": "workspace_name", "expression": "`workspace_name`"},
                                        {"name": "input_tokens", "expression": "`input_tokens`"},
                                        {"name": "output_tokens", "expression": "`output_tokens`"},
                                        {"name": "total_tokens", "expression": "`total_tokens`"},
                                        {"name": "request_count", "expression": "`request_count`"},
                                        {"name": "proxied_cost_usd", "expression": "`proxied_cost_usd`"}
                                    ],
                                    "disaggregated": True
                                }
                            }
                        ],
                        "spec": {
                            "version": 2,
                            "frame": {"showTitle": True, "title": "Top Users by Token Usage"},
                            "rowsPerPage": 50,
                            "widgetType": "table",
                            "encodings": {
                                "columns": [
                                    {"fieldName": "requester", "displayName": "User"},
                                    {"fieldName": "model", "displayName": "Model"},
                                    {"fieldName": "workspace_name", "displayName": "Workspace"},
                                    {"fieldName": "input_tokens", "displayName": "Input Tokens"},
                                    {"fieldName": "output_tokens", "displayName": "Output Tokens"},
                                    {"fieldName": "total_tokens", "displayName": "Total Tokens"},
                                    {"fieldName": "request_count", "displayName": "Requests"},
                                    {"fieldName": "proxied_cost_usd", "displayName": "Proxied Cost ($)"}
                                ]
                            }
                        }
                    },
                    "position": {"x": 0, "y": 2, "width": 12, "height": 10}
                }
            ],
            "pageType": "PAGE_TYPE_CANVAS",
            "layoutVersion": "GRID_V1"
        }
    ]
}

# Check if dashboard already exists
existing = requests.get(f"{host}/api/2.0/lakeview/dashboards", headers=headers, params={"page_size": 100}).json()
dashboard_id = None
for d in existing.get("dashboards", []):
    if d.get("display_name") == dashboard_name and d.get("lifecycle_state") != "TRASHED":
        dashboard_id = d["dashboard_id"]
        print(f"Found existing dashboard: {dashboard_id}")
        break

if dashboard_id:
    resp = requests.patch(
        f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}",
        headers=headers,
        json={"display_name": dashboard_name, "warehouse_id": warehouse_id, "serialized_dashboard": json.dumps(dashboard_json)}
    )
else:
    resp = requests.post(
        f"{host}/api/2.0/lakeview/dashboards",
        headers=headers,
        json={"display_name": dashboard_name, "warehouse_id": warehouse_id, "parent_path": "/Workspace/Users/steve.shao@databricks.com", "serialized_dashboard": json.dumps(dashboard_json)}
    )

result = resp.json()
dashboard_id = result.get("dashboard_id", dashboard_id)
print(f"Dashboard ID: {dashboard_id}")
print(f"Status: {resp.status_code}")
if resp.status_code >= 400:
    print(f"Error: {result}")

Dashboard ID: 01f1333dec521684b38bccf005ced954
Status: 200


In [0]:
# Publish the dashboard
publish_resp = requests.post(
    f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}/published",
    headers=headers,
    json={"warehouse_id": warehouse_id, "embed_credentials": True}
)
print(f"Publish status: {publish_resp.status_code}")
print(f"Dashboard URL: {host}/dashboardsv3/{dashboard_id}/published")
print("Done!")

Publish status: 200
Dashboard URL: https://e2-demo-field-eng.cloud.databricks.com/dashboardsv3/01f1333dec521684b38bccf005ced954/published
Done!
